# Four-way comparison: original · bilinear · bicubic · SEN2SR

Uses **matched triplets** (same as `water_index_hr_comparison.ipynb`):

- **Original** — `SDS_performance_analysis/.../sat_images/<site>/<rel>`
- **Bilinear / bicubic** — `change_tiff_resolution/data/sat_images/<site>/bilinear|bicubic/<rel>`

**Row 1** — same as before: native crops (`128×128` @ ~10 m, `256×256` @ ~5 m from your ×2 pipeline, SEN2SR @ ~2.5 m from one forward, no `predict_large`).

**Row 2** — same 2.5 m grid alignment as before, but the crop is centered on the **shoreline** when possible: **NDWI (Green−NIR) + Otsu** on the **SEN2SR** stack, water/land boundary from a 3×3 erosion, centroid of boundary pixels as the zoom center (falls back to geometric center if the patch is all water or all land). Use `--zoom2-focus center` to force the old center crop.

Implementation: **`opensr_fourway.py`**. `--zoom2-frac` (default `0.2`) = crop size vs frame; `--zoom2-focus shoreline|center`.

```bash
cd change_tiff_resolution
python run_opensr_four_way.py --pick first
python run_opensr_four_way.py --zoom2-frac 0.15 --zoom2-focus shoreline
```

Default PNG: `opensr_outputs/four_way_original_bilinear_bicubic_sen2sr.png`.

In [ ]:
# %pip install mlstac torch rasterio matplotlib numpy

In [1]:
import sys
from pathlib import Path

try:
    PROJECT_ROOT = Path(__file__).resolve().parent
except NameError:
    PROJECT_ROOT = Path.cwd()

if str(PROJECT_ROOT.resolve()) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT.resolve()))

from opensr_fourway import iter_s2_triplets, save_four_way_figure

highres_root = PROJECT_ROOT / "data" / "sat_images"
original_root = (
    PROJECT_ROOT.parent / "SDS_performance_analysis" / "data" / "sat_images"
)

scenes = iter_s2_triplets(highres_root, original_root)
print("S2 triplets:", len(scenes))
if not scenes:
    raise RuntimeError("No S2 triplets — check highres_root and original_root.")

scene = scenes[0]
print("Using:", scene.site, scene.rel)

S2 triplets: 958
Using: australianarrabeen S2\S2_20151218_235302.tif


In [2]:
out = save_four_way_figure(scene, PROJECT_ROOT)
print("Saved:", out.resolve())

Saved: C:\Users\jnicolow\Documents\research\CRC\change_tiff_resolution\opensr_outputs\four_way_original_bilinear_bicubic_sen2sr.png
